# 稠密连接网络（DenseNet）
---
## 环境配置

In [1]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["TILE_FWK_DEVICE_ID"] = "1"
import numpy as np
import pypto
import torch
import torch_npu
import torchvision
from torch import nn
device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
mode = pypto.RunMode.NPU
from src.PyPTOConv2DModule import PyPTOConv2d
from src.PyPTOPoolModule import PyPTOMaxPool2d, PyPTOAvgPool2d
from src.PyPTOLinearFuseModule import PyPTOLinear
from src.D2LFunction import *
from src.AnswerUtlis import *
from src import Models
from collections import OrderedDict
import torchinfo

---

&emsp;&emsp;DenseNet 是由若干 `DenseBlock` + `Transition` 串联而成。下面以 `densenet121` 为例展示其网络结构。

In [2]:
def conv_block(num_channels):
        return nn.Sequential(
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.LazyConv2d(num_channels, kernel_size=3, padding=1))

class DenseBlock(nn.Module):
    def __init__(self, num_convs, num_channels):
        super().__init__()
        layer = []
        for i in range(num_convs):
            layer.append(conv_block(num_channels))
        self.net = nn.Sequential(*layer)
    def forward(self, x):
        for blk in self.net:
            y = blk(x)
            x = torch.cat((x, y), dim=1)
        return x

class DenseNet(nn.Module):
    def __init__(self, num_channels=64, growth_rate=32, arch=(4, 4, 4, 4),
                 num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.LazyBatchNorm2d(), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        for i, num_convs in enumerate(arch):
            self.net.add_module(f'dense_blk{i+1}', DenseBlock(num_convs, growth_rate))
            num_channels += num_convs * growth_rate
            if i != len(arch) - 1:
                self.net.add_module(
                    f'tran_blk{i+1}',
                    nn.Sequential(
                        nn.LazyBatchNorm2d(), nn.ReLU(),
                        nn.LazyConv2d(num_channels // 2, kernel_size=1),
                        nn.AvgPool2d(kernel_size=2, stride=2)))
                num_channels = num_channels // 2
        self.net.add_module(
            'last', nn.Sequential(
                nn.LazyBatchNorm2d(), nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
                nn.LazyLinear(num_classes)))
    def forward(self, x):
        return self.net(x)

model = DenseNet(num_classes=10)
x = torch.randn(2, 1, 224, 224)
print('DenseNet output shape:', model(x).shape)
print('DenseNet (pure torch) defined')

DenseNet output shape: torch.Size([2, 10])
DenseNet (pure torch) defined


In [3]:
Densenet_pypto = Models.Densenet_pypto(in_channels=1, num_classes=10).to(device)
torchinfo.summary(Densenet_pypto, (1, 1, 96, 96), device=device)  # device 显式指定, 否则 torchinfo 默认 CPU 导致 PyPTO kernel 报 Not npu device

Layer (type:depth-idx)                        Output Shape              Param #
Densenet_pypto                                [1, 10]                   --
├─Sequential: 1-1                             [1, 64, 24, 24]           --
│    └─PyPTOConv2d: 2-1                       [1, 64, 48, 48]           3,200
│    └─BatchNorm2d: 2-2                       [1, 64, 48, 48]           128
│    └─ReLU: 2-3                              [1, 64, 48, 48]           --
│    └─PyPTOMaxPool2d: 2-4                    [1, 64, 24, 24]           --
├─Sequential: 1-2                             [1, 248, 3, 3]            --
│    └─DenseBlock_pypto: 2-5                  [1, 192, 24, 24]          --
│    │    └─Sequential: 3-1                   --                        130,048
│    └─Sequential: 2-6                        [1, 96, 12, 12]           --
│    │    └─BatchNorm2d: 3-2                  [1, 192, 24, 24]          384
│    │    └─ReLU: 3-3                         [1, 192, 24, 24]          --
│    │    

---
## 练习 7.7.1

为什么在过渡层要使用平均汇聚而不是最大汇聚？

### 解答

&emsp;&emsp;与 ResNet / DenseNet 类似，稠密块通过跨层连接扩展目标函数的 $f$；过渡层使用平均池化：

* 平均池通过对特征图所有元素取平均，可以保留背景信息；最大池只关注最显著的特征，可能丢失上下文；
* 平均池降低了模型的复杂度，有助于缓解过拟合。

在稠密层使用平均池可以在降维时更好地保留信息。

---
## 练习 7.7.2

DenseNet 的一个优点是模型参数比 ResNet 小很多，为什么？

### 解答

&emsp;&emsp;因为 DenseNet 中每个卷积层的输出通道数都很小（growth_rate 12~32），仅靠密集连接就能使最终通道数很大，同时参数量小很多。BN 参数、全连接层参数也比 ResNet 少。

---
## 练习 7.7.3

DenseNet 在高复杂度的数据集上可能存在显存消耗过大的问题。
1. 为什么会这样？可以将输入形状改为 $224\times224$ 来观察实际的显存消耗吗？
2. 是否有其他方法来减少显存消耗？

### 解答

**第1问：**

&emsp;&emsp;DenseNet 显存消耗大的根本原因在于**中间特征图的显存占用**而非参数量：

1. DenseNet 每个卷积层的输出通道数很小（growth_rate 一般为 12~32），但稠密连接要求把所有前置层的输出**沿通道方向拼接**，因此越靠近网络尾部的稠密块，输入通道数越大（例如 densenet121 最后一个稠密块的输入通道数为 1024）。中间特征图的显存占用随网络加深而急剧增长；
2. 反向传播时每一层的激活值都需要保留用于计算梯度（不可释放），进一步放大显存占用；
3. 与 ResNet 相比，ResNet 每个残差块的通道数独立增长（64~512），不会跨层累计，因此在相同参数量下 DenseNet 的中间特征图显存占用显著更高；
4. 输入形状增大（如 $224	imes224$）时，特征图的空间分辨率更高，激活值显存占用随输入面积近似线性增长，对显存压力更大。

&emsp;&emsp;因此，训练 DenseNet 时常见的做法是控制输入尺寸与批量大小，并在显存受限时改用 DenseNet-BC 等参数/特征图更紧凑的变体。

**第2问：**

&emsp;&emsp;减小输入图片尺寸或减小批量大小都可以降低显存占用。此外可采用以下策略：

1. 减少模型复杂度：使用更小的 DenseNet 变体（如 DenseNet-BC）或自定义更小的网络；
2. 模型压缩：权重剪枝、知识蒸馏等；
3. 更换优化器：Adam 等会存储更多梯度状态，可改用 SGD；
4. 混合精度训练：使用 float16 + float32 混合精度；
5. 梯度累积：用多个小 batch 模拟大 batch 效果。

---
## 练习 7.7.4

实现 DenseNet 论文中表1所示的不同 DenseNet 版本。

### 解答

&emsp;&emsp;参考原文 https://arxiv.org/abs/1608.06993 表1，分别构建 `densenet121`、`densenet169`、`densenet201`、`densenet161`：

<div style="border: solid 16px #f1f1f8; text-align: center; background-color: #f6f7f9">
<img src="../images/ch07-1-6-DenseNet.png" style="width: 400px">
<p style="margin: 12px 0 4px 0; font-size: 0.9em; color: #555; text-align: center;">DenseNet不同版本结构表</p>
</div>
<br />

使用 `torch` 编程进行验证：

In [4]:
class _DenseLayer(nn.Sequential):
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super().__init__()
        self.add_module('norm1', nn.BatchNorm2d(num_input_features))
        self.add_module('relu1', nn.ReLU(inplace=True))
        self.add_module('conv1', nn.Conv2d(num_input_features, bn_size * growth_rate, kernel_size=1, stride=1, bias=False))
        self.add_module('norm2', nn.BatchNorm2d(bn_size * growth_rate))
        self.add_module('relu2', nn.ReLU(inplace=True))
        self.add_module('conv2', nn.Conv2d(bn_size * growth_rate, growth_rate, kernel_size=3, stride=1, padding=1, bias=False))
        self.drop_rate = drop_rate
    def forward(self, x):
        new_features = super().forward(x)
        if self.drop_rate > 0:
            new_features = F.dropout(new_features, p=self.drop_rate, training=self.training)
        return torch.cat([x, new_features], 1)

net = Models.densenet121(num_classes=10).to(device)
x = torch.randn(1, 3, 224, 224, device=device)
print('DenseNet-121 output shape:', net(x).shape)

DenseNet-121 output shape: torch.Size([1, 10])


使用 `PyPTO` 编程进行验证：

In [5]:
net_pypto_121 = Models.densenet121_pypto(num_classes=10).to(device)
net_pypto_169 = Models.densenet169_pypto(num_classes=10).to(device)
net_pypto_201 = Models.densenet201_pypto(num_classes=10).to(device)
net_pypto_161 = Models.densenet161_pypto(num_classes=10).to(device)

x = torch.randn(1, 3, 224, 224, device=device)
print('densenet121_pypto output shape:', net_pypto_121(x).shape)
print('densenet169_pypto output shape:', net_pypto_169(x).shape)
print('densenet201_pypto output shape:', net_pypto_201(x).shape)
# densenet161 通道最宽 (growth 48), 224 输入实测 ALLOC_FAILED 显存不足, 用 96 输入验证
# 注意: 121/169/201 三个模型同时驻留 NPU 显存后, 161@96 仍实测 ALLOC_FAILED,
# 故先释放前 3 个模型再构造 161 (验证 161 单独前向能力)。
del net_pypto_121, net_pypto_169, net_pypto_201
torch.npu.empty_cache()
x161 = torch.randn(1, 3, 96, 96, device=device)
print('densenet161_pypto output shape:', net_pypto_161(x161).shape)

densenet121_pypto output shape: torch.Size([1, 10])


densenet169_pypto output shape: torch.Size([1, 10])


densenet201_pypto output shape: torch.Size([1, 10])


densenet161_pypto output shape: torch.Size([1, 10])


---
## 练习 7.7.5

应用DenseNet的思想构建一个用于多标签学习的模型。可以参考4.10节中的房价预测方法。

### 解答

&emsp;&emsp;参考 4.10 节中的房价预测方法，这是一个传统的机器学习任务，实现步骤包括：加载 Kaggle 房价数据集、K 折交叉验证、训练模型并评估。以下答案实现数据集的下载与加载（纯 torch / pandas）；基于 DenseNet 思想的模型构建、
K 折交叉验证训练与评估步骤与 4.10 节房价预测一致，此处不再重复实现。

使用 `torch` 编程进行验证：

In [6]:
import os
import urllib.request
import pandas as pd

data_root = '../data/kaggle_house'
os.makedirs(data_root, exist_ok=True)

def download(fname):
    url = 'https://d2l-data.s3-accelerate.amazonaws.com/' + fname
    path = os.path.join(data_root, fname)
    if not os.path.exists(path):
        print('downloading', fname, '...')
        urllib.request.urlretrieve(url, path)
    return path

class KaggleHouse:
    def __init__(self, batch_size, train=None, val=None):
        self.batch_size = batch_size
        if train is None:
            self.raw_train = pd.read_csv(download('kaggle_house_pred_train.csv'))
            self.raw_val = pd.read_csv(download('kaggle_house_pred_test.csv'))
    def preprocess(self):
        print('train data shape:', self.raw_train.shape)
        print('val data shape:', self.raw_val.shape)

data = KaggleHouse(batch_size=64)
data.preprocess()
print('data prepared')

train data shape: (1460, 81)
val data shape: (1459, 80)
data prepared


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#](https://datawhalechina.github.io/d2l-ai-solutions-manual/#)